<a href="https://colab.research.google.com/github/vs-152/FL-Contributions-Incentives-Project/blob/main/ISO_CIFAR10_OR_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ─────────────────────────────────────────────────────────────
#  Imports
# ─────────────────────────────────────────────────────────────
import os
import copy
import time
import glob
import shutil
import tempfile
from itertools import chain, combinations

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.utils.data.sampler import SubsetRandomSampler
from sklearn.model_selection import StratifiedShuffleSplit
from scipy.special import comb
import matplotlib.pyplot as plt
from tqdm import tqdm
import nibabel as nib
import pulp
import onnxruntime
import random

# ─────────────────────────────────────────────────────────────
#  MONAI
# ─────────────────────────────────────────────────────────────
from monai.config import print_config
from monai.utils import set_determinism
from monai.data import CacheDataset, DataLoader, decollate_batch
from monai.handlers.utils import from_engine
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.networks.nets import SegResNet
from monai.apps import DecathlonDataset
from monai.transforms import (
    Activations,
    Activationsd,
    AsDiscrete,
    AsDiscreted,
    Compose,
    EnsureChannelFirstd,
    EnsureTyped,
    Invertd,
    LoadImaged,
    MapTransform,
    NormalizeIntensityd,
    Orientationd,
    RandFlipd,
    RandScaleIntensityd,
    RandShiftIntensityd,
    RandSpatialCropd,
    ScaleIntensityd,
    Spacingd,
    SelectItemsd
)

# ─────────────────────────────────────────────────────────────
#  Custom Modules
# ─────────────────────────────────────────────────────────────
from utils import *

# ─────────────────────────────────────────────────────────────
#  Device & Setup
# ─────────────────────────────────────────────────────────────
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print_config()
set_determinism(seed=0)


2026-03-06 13:20:05.274703605 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


MONAI version: 1.6.dev2542
Numpy version: 2.1.2
Pytorch version: 2.8.0+cu126
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: 612f3dd3cba4d73cfcea4b5329079e20aa31523d
MONAI __file__: /home/<username>/miniconda3/envs/m_quant_py310/lib/python3.10/site-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: NOT INSTALLED or UNKNOWN VERSION.
ITK version: 5.4.4
Nibabel version: 5.3.2
scikit-image version: 0.25.2
scipy version: 1.15.3
Pillow version: 11.0.0
Tensorboard version: NOT INSTALLED or UNKNOWN VERSION.
gdown version: NOT INSTALLED or UNKNOWN VERSION.
TorchVision version: 0.23.0+cu126
tqdm version: 4.67.1
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 7.0.0
pandas version: 2.3.2
einops version: 0.8.1
transformers version: NOT INSTALLED or UNKNOWN VERSION.
mlflow version: NOT INSTALLED or UNKNOWN VERSION.
pynrrd version: NOT INSTALLED or UNKNOWN VERSION.
clearml version: NOT INSTALLED or UNKNOWN VERSION.

For d

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# -----------------------------------------------------------
# 0. paths & meta-data (unchanged) ---------------------------
# -----------------------------------------------------------
BRATS_DIR = "/mnt/d/Datasets/FETS_data/MICCAI_FeTS2022_TrainingData"
CSV_PATH  = f"{BRATS_DIR}/partitioning_1.csv"
MODALITIES = ["flair", "t1", "t1ce", "t2"]
LABEL_KEY  = "seg"

# -----------------------------------------------------------
# 1. read partition file  ➜  { id : [subjects] } ------------
# -----------------------------------------------------------
part_df = pd.read_csv(CSV_PATH)

# --- compute subject counts per site -----------------------
site_counts = (
    part_df.groupby("Partition_ID")["Subject_ID"]
           .nunique()
)

TOP_K = 6  # keep 6 most populated sites for training

# site IDs for training (top-K by subject count)
TRAIN_CENTRES = set(
    site_counts.sort_values(ascending=False)
               .head(TOP_K)
               .index.tolist()
)

# everything else is validation
VAL_CENTRES = set(site_counts.index) - TRAIN_CENTRES

print("Train centres (top 6 by subject count):")
print(site_counts.loc[sorted(TRAIN_CENTRES)])
print("\nValidation centres (remaining):")
print(site_counts.loc[sorted(VAL_CENTRES)])

# map centre → list of subject IDs
partition_map = (
    part_df.groupby("Partition_ID")["Subject_ID"]
           .apply(list).to_dict()
)

# split once, reuse everywhere
train_partitions = {
    cid: sids for cid, sids in partition_map.items()
    if cid in TRAIN_CENTRES
}
val_subjects = sum((partition_map[cid] for cid in VAL_CENTRES), [])


# -----------------------------------------------------------
# 2. helper to build MONAI-style record dicts ----------------
# -----------------------------------------------------------
def build_records(subject_ids):
    recs = []
    for sid in subject_ids:
        sdir = f"{BRATS_DIR}/{sid}"
        images = [f"{sdir}/{sid}_{m}.nii.gz" for m in MODALITIES]  # 4 modalities
        recs.append({"image": images, "label": f"{sdir}/{sid}_{LABEL_KEY}.nii.gz"})
    return recs


# -----------------------------------------------------------
# 3. MONAI CacheDatasets ------------------------------------
# -----------------------------------------------------------
FRAC, SEED = 1, 42   # FRAC for subsampling within each site
rng = random.Random(SEED)

train_datasets = {}
for cid, subj_ids in train_partitions.items():
    k = max(1, int(len(subj_ids) * FRAC))   # e.g. 0.3 for 30% subsample
    sample_ids = rng.sample(subj_ids, k)
    train_datasets[cid] = CacheDataset(
        build_records(sample_ids), transform=train_transform, cache_rate=1
    )

Train centres (top 6 by subject count):
Partition_ID
1     511
4      47
6      34
13     35
18    382
21     35
Name: Subject_ID, dtype: int64

Validation centres (remaining):
Partition_ID
2      6
3     15
5     22
7     12
8      8
9      4
10     8
11    14
12    11
14     6
15    13
16    30
17     9
19     4
20    33
22     7
23     5
Name: Subject_ID, dtype: int64


NameError: name 'train_transform' is not defined

#  SimCLR model

✅ Define Encoder + Projection Head

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SimCLR Model (2D)
# ─────────────────────────────────────────────────────────────
from monai.networks.nets import resnet

class SimCLR2D(nn.Module):
    def __init__(self, projection_dim=128):
        super().__init__()

        # 2D ResNet18
        self.encoder = resnet.resnet18(
            spatial_dims=2,
            n_input_channels=4,  # 4 MRI modalities
            num_classes=512
        )

        self.encoder.fc = nn.Identity()

        self.projector = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, projection_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return h, z


model = SimCLR2D().to(device)

In [ ]:
model.load_state_dict(torch.load("./simclr_models/simclr_2d_best.pt"))
model.eval()

# Build Per-Site Embedding Extractor

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SimCLR Contrastive Transform (image only)
# ─────────────────────────────────────────────────────────────

from monai.transforms import RandRotate90d, RandGaussianNoised

simclr_2d_transform = Compose(
    [
        EnsureTyped(keys="image"),
        RandFlipd(keys="image", prob=0.5, spatial_axis=0),
        RandFlipd(keys="image", prob=0.5, spatial_axis=1),
        RandRotate90d(keys="image", prob=0.5),
        # RandScaleIntensityd(keys="image", factors=0.1, prob=0.8),
        # RandShiftIntensityd(keys="image", offsets=0.1, prob=0.8),
        # RandGaussianNoised(keys="image", prob=0.5),
    ]
)

class SimCLR2DDataset(Dataset):
    def __init__(self, monai_dataset, transform):
        self.dataset = monai_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        data = self.dataset[idx]
        img = data["image"]  # [C, H, W, D]

        # Random axial slice
        depth = img.shape[-1]
        z = torch.randint(0, depth, (1,)).item()
        slice_2d = img[..., z]  # [C, H, W]

        # Wrap as dict for MONAI transforms
        slice_dict = {"image": slice_2d}

        view1 = self.transform(slice_dict)["image"]
        view2 = self.transform(slice_dict)["image"]

        return view1, view2
        
simclr_train_datasets = {
    cid: SimCLR2DDataset(ds, simclr_2d_transform)
    for cid, ds in train_datasets.items()
}


def extract_site_embeddings(site_dataset, model, device):
    loader = DataLoader(
        site_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2
    )

    all_embeddings = []

    with torch.no_grad():
        for v1, _ in loader:   # only need one view
            v1 = v1.to(device)
            h, _ = model(v1)
            all_embeddings.append(h.cpu())

    return torch.cat(all_embeddings, dim=0)

site_embeddings = {}

for cid, dataset in simclr_train_datasets.items():
    print(f"Extracting embeddings for site {cid}")
    site_embeddings[cid] = extract_site_embeddings(dataset, model, device)

In [ ]:
all_embs = []
labels = []

for i, (cid, emb) in enumerate(site_embeddings.items()):
    all_embs.append(emb)
    labels.extend([i] * emb.shape[0])

all_embs = torch.cat(all_embs).numpy()
labels = np.array(labels)

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
proj = pca.fit_transform(all_embs)

plt.figure(figsize=(8,6))
for i, cid in enumerate(site_embeddings.keys()):
    mask = labels == i
    plt.scatter(proj[mask, 0], proj[mask, 1], s=5, label=str(cid))

plt.legend()
plt.title("PCA of SimCLR Embeddings by Site")
plt.savefig("./plots/pca_plot.pdf", format="pdf", bbox_inches="tight")
plt.show()